In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# CAFA6 GlyphMatics GlyphString Combo Layer 🧬

This enhanced notebook keeps the original CAFA6 `submission.tsv` format while adding a reversible GlyphMatics protein sequence channel:

1. **Protein → glyphs**
2. **Glyphs → grouped glyphstrings**
3. **Glyphstrings → reconstructed proteins**
4. **Roundtrip validation**
5. **Low-amplitude glyph-derived combo calibration**
6. **GO DAG propagation + competition-safe output**

The final `submission.tsv` contains only normal CAFA rows:

```text
protein_id<TAB>GO_term<TAB>score
```

Sidecar files are written for audit/combo testing:
- `glyph_combo_report.json`
- `glyphstrings_sample.tsv`
- `glyph_features.tsv`
- `glyph_combo_outputs/*.tsv`


In [ ]:

# ============================================================
# CAFA6 + GlyphMatics Protein GlyphString Roundtrip Combos
# Submission-safe notebook cell
#
# Pipeline:
#   1) Load GOA + ProtT5/interpro predictions
#   2) Load GO ontology
#   3) Optionally load test protein sequences
#   4) Translate protein sequences -> glyphs -> glyphstrings
#   5) Translate glyphstrings back -> protein sequences
#   6) Use roundtrip-verified glyph features for low-risk combo calibration
#   7) Propagate through GO DAG and write submission.tsv
#
# Final output remains Kaggle/CAFA compliant:
#   protein_id<TAB>GO_term<TAB>score
# ============================================================

import os
import re
import json
import math
import gzip
import hashlib
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd


# ----------------------------
# Deterministic file discovery
# ----------------------------

def find_first(root="/kaggle/input", names=None, contains=None, suffixes=None):
    """
    Robust Kaggle input discovery.
    - names: exact filename allowlist
    - contains: list of lowercase substrings that must appear in basename
    - suffixes: allowed suffixes such as [".tsv", ".obo", ".fasta"]
    """
    root = Path(root)
    names = set(names or [])
    contains = [s.lower() for s in (contains or [])]
    suffixes = tuple(suffixes or [])
    hits = []
    if not root.exists():
        return None

    for p in root.rglob("*"):
        if not p.is_file():
            continue
        b = p.name.lower()
        ok = True
        if names and p.name not in names:
            ok = False
        if contains and not all(s in b for s in contains):
            ok = False
        if suffixes and not str(p).lower().endswith(suffixes):
            ok = False
        if ok:
            hits.append(p)

    if not hits:
        return None

    # Prefer shorter paths, then lexicographic for deterministic behavior.
    hits = sorted(hits, key=lambda x: (len(str(x)), str(x)))
    return str(hits[0])


def open_text_maybe_gzip(path):
    path = str(path)
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")
    return open(path, "r", encoding="utf-8", errors="replace")


# ----------------------------
# GO Ontology
# ----------------------------

class GOOntology:
    ROOTS = {"GO:0003674", "GO:0008150", "GO:0005575"}

    def __init__(self, obo_path):
        self.parents = defaultdict(set)
        self.children = defaultdict(set)
        self.namespace = {}
        self.anc_cache = {}
        self.obo_path = obo_path
        self._parse_obo(obo_path)

    def _parse_obo(self, obo_path):
        if not obo_path or not Path(obo_path).exists():
            raise FileNotFoundError(f"go-basic.obo not found: {obo_path}")

        tid = None
        obsolete = False
        with open_text_maybe_gzip(obo_path) as f:
            for raw in f:
                line = raw.strip()

                if line == "[Term]":
                    tid = None
                    obsolete = False
                    continue

                if line.startswith("id: GO:"):
                    tid = line[4:].strip()
                    obsolete = False
                    continue

                if not tid:
                    continue

                if line == "is_obsolete: true":
                    obsolete = True
                    continue

                if obsolete:
                    continue

                if line.startswith("namespace: "):
                    self.namespace[tid] = line.split("namespace: ", 1)[1].strip()

                elif line.startswith("is_a: "):
                    parent = line.split()[1]
                    self.parents[tid].add(parent)
                    self.children[parent].add(tid)

                elif line.startswith("relationship: part_of "):
                    parent = line.split()[2]
                    self.parents[tid].add(parent)
                    self.children[parent].add(tid)

    def ancestors(self, term):
        if term in self.anc_cache:
            return self.anc_cache[term]
        seen = set()
        stack = list(self.parents.get(term, []))
        while stack:
            p = stack.pop()
            if p in seen:
                continue
            seen.add(p)
            stack.extend(self.parents.get(p, []))
        self.anc_cache[term] = seen
        return seen


# ----------------------------
# Prediction IO
# ----------------------------

def load_preds(path):
    """
    Reads CAFA-style TSV:
      protein_id<TAB>GO:xxxxxxx<TAB>score
    Keeps max score per protein/term.
    """
    out = defaultdict(dict)
    if not path or not Path(path).exists():
        print(f"[WARN] Missing prediction file: {path}")
        return out

    bad = 0
    with open_text_maybe_gzip(path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                bad += 1
                continue
            p, t, s = parts[0], parts[1], parts[2]
            try:
                s = float(s)
            except Exception:
                bad += 1
                continue
            if not t.startswith("GO:"):
                bad += 1
                continue
            if t not in out[p] or s > out[p][t]:
                out[p][t] = s

    print(f"[LOAD] {Path(path).name}: proteins={len(out):,}, bad_lines={bad:,}")
    return out


def save_preds(preds, path, threshold=0.001):
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for protein_id in sorted(preds):
            terms = preds[protein_id]
            for term, score in sorted(terms.items(), key=lambda x: (-x[1], x[0])):
                if score >= threshold:
                    f.write(f"{protein_id}\t{term}\t{float(score):.6f}\n")
                    n += 1
    print(f"[SAVE] {path}: rows={n:,}")


def ensemble(goa, prott5, w_goa=0.55):
    w_pt5 = 1.0 - w_goa
    out = {}
    for p in sorted(set(goa) | set(prott5)):
        a = goa.get(p, {})
        b = prott5.get(p, {})
        r = {}
        for t in set(a) | set(b):
            s = w_goa * a.get(t, 0.0) + w_pt5 * b.get(t, 0.0)
            if s > 0:
                r[t] = float(s)
        out[p] = r
    return out


# ----------------------------
# Protein <-> GlyphString Codec
# ----------------------------

class ProteinGlyphCodec:
    """
    Reversible amino-acid/glyph codec.

    Design:
    - Maps A-Z plus common protein sentinels to single Unicode Braille glyphs.
    - Glyphstrings are grouped for motif/chunk combo testing.
    - Roundtrip is exact for all mapped symbols.
    - Unmapped symbols are encoded as escaped hex packets, so the codec remains reversible.
    """

    SEP = "⫶"
    ESC_OPEN = "⟦"
    ESC_CLOSE = "⟧"

    def __init__(self):
        alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ*-_."
        # Braille block gives compact single-codepoint visual glyphs.
        glyphs = [chr(0x2801 + i) for i in range(len(alphabet))]
        self.a2g = dict(zip(alphabet, glyphs))
        self.g2a = {v: k for k, v in self.a2g.items()}
        self.alphabet = alphabet

    def protein_to_glyphs(self, seq):
        seq = str(seq).strip().upper()
        out = []
        for ch in seq:
            if ch in self.a2g:
                out.append(self.a2g[ch])
            else:
                out.append(f"{self.ESC_OPEN}{ord(ch):04X}{self.ESC_CLOSE}")
        return "".join(out)

    def glyphs_to_protein(self, glyphs):
        glyphs = str(glyphs).replace(self.SEP, "")
        out = []
        i = 0
        while i < len(glyphs):
            ch = glyphs[i]
            if ch == self.ESC_OPEN:
                j = glyphs.find(self.ESC_CLOSE, i + 1)
                if j == -1:
                    raise ValueError("Broken glyph escape packet")
                hx = glyphs[i + 1:j]
                out.append(chr(int(hx, 16)))
                i = j + 1
            else:
                if ch not in self.g2a:
                    raise ValueError(f"Unknown glyph {repr(ch)} at position {i}")
                out.append(self.g2a[ch])
                i += 1
        return "".join(out)

    def protein_to_glyphstring(self, seq, chunk=5):
        glyphs = self.protein_to_glyphs(seq)
        if chunk <= 0:
            return glyphs
        return self.SEP.join(glyphs[i:i + chunk] for i in range(0, len(glyphs), chunk))

    def glyphstring_to_protein(self, glyphstring):
        return self.glyphs_to_protein(glyphstring)

    def checksum(self, seq):
        seq = str(seq).strip().upper()
        return hashlib.blake2s(seq.encode("utf-8"), digest_size=8).hexdigest()

    def signature(self, seq, chunk=5, edge=3):
        glyphstring = self.protein_to_glyphstring(seq, chunk=chunk)
        chunks = glyphstring.split(self.SEP) if glyphstring else []
        head = self.SEP.join(chunks[:edge])
        tail = self.SEP.join(chunks[-edge:]) if len(chunks) > edge else ""
        return {
            "aa_len": len(seq),
            "checksum": self.checksum(seq),
            "glyph_head": head,
            "glyph_tail": tail,
            "glyph_chunks": len(chunks),
        }


# ----------------------------
# FASTA loading + glyph tests
# ----------------------------

def load_fasta(path):
    seqs = {}
    if not path or not Path(path).exists():
        print(f"[WARN] Missing FASTA: {path}")
        return seqs

    current = None
    buf = []
    with open_text_maybe_gzip(path) as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current is not None:
                    seqs[current] = "".join(buf).upper()
                # CAFA target IDs are normally first token after ">"
                current = line[1:].split()[0]
                buf = []
            else:
                buf.append(line)
        if current is not None:
            seqs[current] = "".join(buf).upper()

    print(f"[FASTA] {Path(path).name}: proteins={len(seqs):,}")
    return seqs


def discover_test_fasta():
    candidates = []

    # High-priority common names.
    common_names = [
        "testsuperset.fasta",
        "testsuperset.fasta.gz",
        "test_sequences.fasta",
        "test_sequences.fasta.gz",
        "test.fasta",
        "test.fasta.gz",
    ]

    for name in common_names:
        p = find_first("/kaggle/input", names=[name])
        if p:
            candidates.append(p)

    # Generic fallback: prefer paths containing Test/test and fasta/fa.
    for p in Path("/kaggle/input").rglob("*"):
        if not p.is_file():
            continue
        lower = str(p).lower()
        if lower.endswith((".fasta", ".fa", ".faa", ".fasta.gz", ".fa.gz", ".faa.gz")):
            score = 0
            if "test" in lower:
                score -= 10
            if "target" in lower:
                score -= 5
            if "train" in lower:
                score += 10
            candidates.append((score, str(p)))

    normalized = []
    for x in candidates:
        if isinstance(x, tuple):
            normalized.append(x)
        else:
            lower = str(x).lower()
            score = 0
            if "test" in lower:
                score -= 10
            if "target" in lower:
                score -= 5
            if "train" in lower:
                score += 10
            normalized.append((score, str(x)))

    if not normalized:
        return None

    normalized = sorted(set(normalized), key=lambda z: (z[0], len(z[1]), z[1]))
    return normalized[0][1]


def roundtrip_combo_test(seqs, codec, sample_limit=1000, chunk_options=(3, 5, 8)):
    """
    Converts protein -> glyphstring -> protein for several chunk layouts.
    Uses the reconstructed protein for downstream feature extraction.
    """
    ids = sorted(seqs)[:sample_limit]
    failures = []
    rows = []

    for protein_id in ids:
        seq = seqs[protein_id]
        for chunk in chunk_options:
            gs = codec.protein_to_glyphstring(seq, chunk=chunk)
            recon = codec.glyphstring_to_protein(gs)
            ok = (recon == seq.upper())
            if not ok:
                failures.append({
                    "protein_id": protein_id,
                    "chunk": chunk,
                    "source_prefix": seq[:50],
                    "recon_prefix": recon[:50],
                })
            sig = codec.signature(recon, chunk=chunk)
            rows.append({
                "protein_id": protein_id,
                "chunk": chunk,
                "roundtrip_ok": ok,
                "aa_len": sig["aa_len"],
                "checksum": sig["checksum"],
                "glyph_chunks": sig["glyph_chunks"],
                "glyph_head": sig["glyph_head"],
                "glyph_tail": sig["glyph_tail"],
            })

    report = {
        "tested_proteins": len(ids),
        "chunk_options": list(chunk_options),
        "rows": len(rows),
        "roundtrip_pass": len(failures) == 0,
        "failure_count": len(failures),
        "failures_sample": failures[:5],
    }
    return report, pd.DataFrame(rows)


# ----------------------------
# Glyph-derived features
# ----------------------------

HYDROPHOBIC = set("AILMFWVY")
POLAR = set("STNQCY")
CHARGED = set("DEKRH")
SPECIAL = set("GP")
AMBIG = set("XBZUOJ")

def shannon_entropy(seq):
    if not seq:
        return 0.0
    c = Counter(seq)
    n = len(seq)
    return -sum((v / n) * math.log2(v / n) for v in c.values())

def max_run_fraction(seq):
    if not seq:
        return 0.0
    best = 1
    cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return best / max(1, len(seq))

def hydrophobic_window_score(seq, window=19):
    if not seq:
        return 0.0
    if len(seq) < window:
        return sum(ch in HYDROPHOBIC for ch in seq) / max(1, len(seq))
    best = 0
    for i in range(0, len(seq) - window + 1):
        best = max(best, sum(ch in HYDROPHOBIC for ch in seq[i:i + window]))
    return best / window

def glyph_features_from_reconstructed(seq, codec):
    """
    Intentionally goes through glyphstring and back before feature extraction.
    This proves that combo testing uses the reversible glyph channel, not a separate shortcut.
    """
    gs = codec.protein_to_glyphstring(seq, chunk=5)
    recon = codec.glyphstring_to_protein(gs)

    n = len(recon)
    if n == 0:
        return {
            "length": 0,
            "entropy_norm": 0.0,
            "hydrophobic_frac": 0.0,
            "polar_frac": 0.0,
            "charged_frac": 0.0,
            "special_frac": 0.0,
            "ambig_frac": 0.0,
            "low_complexity": 0.0,
            "tm_window": 0.0,
            "glyph_checksum": codec.checksum(recon),
        }

    entropy_norm = shannon_entropy(recon) / math.log2(26)
    return {
        "length": n,
        "entropy_norm": float(max(0.0, min(1.0, entropy_norm))),
        "hydrophobic_frac": sum(ch in HYDROPHOBIC for ch in recon) / n,
        "polar_frac": sum(ch in POLAR for ch in recon) / n,
        "charged_frac": sum(ch in CHARGED for ch in recon) / n,
        "special_frac": sum(ch in SPECIAL for ch in recon) / n,
        "ambig_frac": sum(ch in AMBIG for ch in recon) / n,
        "low_complexity": max_run_fraction(recon),
        "tm_window": hydrophobic_window_score(recon),
        "glyph_checksum": codec.checksum(recon),
    }


def build_glyph_feature_table(seqs, codec, ids=None):
    ids = sorted(ids or seqs.keys())
    rows = []
    for protein_id in ids:
        if protein_id not in seqs:
            continue
        feat = glyph_features_from_reconstructed(seqs[protein_id], codec)
        feat["protein_id"] = protein_id
        rows.append(feat)
    return pd.DataFrame(rows)


def glyph_calibrate(preds, seqs, codec, glyph_weight=0.012):
    """
    Low-risk score calibration from roundtrip-verified glyph features.
    It does not invent GO terms. It only rescales existing model terms slightly.

    Why small? CAFA predictions are sensitive; glyph features are used as a
    deterministic combo signal, not as a replacement model.
    """
    if glyph_weight <= 0 or not seqs:
        return preds

    out = {}
    for protein_id, terms in preds.items():
        seq = seqs.get(protein_id)
        if not seq:
            out[protein_id] = terms
            continue

        f = glyph_features_from_reconstructed(seq, codec)

        # Composite confidence from reconstructed protein sequence.
        # Centered around 0.0; clipped to avoid damaging strong base predictions.
        structure_signal = (
            +0.45 * (f["entropy_norm"] - 0.62)
            +0.25 * (f["tm_window"] - 0.48)
            -0.20 * max(0.0, f["low_complexity"] - 0.10)
            -0.10 * f["ambig_frac"]
        )
        scale = 1.0 + glyph_weight * max(-1.0, min(1.0, structure_signal))

        new_terms = {}
        for term, score in terms.items():
            # Keep roots unchanged; they are set explicitly after propagation.
            if term in GOOntology.ROOTS:
                new_terms[term] = score
            else:
                new_terms[term] = float(max(0.0, min(1.0, score * scale)))
        out[protein_id] = new_terms

    return out


# ----------------------------
# Propagation and combo scoring
# ----------------------------

def propagate(preds, ont, alpha=0.70, power=0.80, max_s=0.93, topk=270):
    out = {}
    for protein_id, scores in preds.items():
        u = dict(scores)

        # Up-propagate each child score to ancestors.
        for term, score in list(scores.items()):
            for anc in ont.ancestors(term):
                if anc not in u or score > u[anc]:
                    u[anc] = score

        # DAG consistency smoothing:
        # child should not overpower its known ancestors too aggressively.
        for term in list(u):
            ancestors = [a for a in ont.ancestors(term) if a in u]
            if ancestors:
                min_parent = min(u[a] for a in ancestors)
                if min_parent < u[term]:
                    u[term] = alpha * min_parent + (1.0 - alpha) * u[term]

        # Per-protein normalization excluding root terms.
        vals = [v for k, v in u.items() if k not in ont.ROOTS]
        if vals:
            mx = max(vals)
            if 0 < mx < max_s:
                for k in list(u):
                    if k not in ont.ROOTS:
                        u[k] = min(1.0, (u[k] / mx) ** power * max_s)

        for root in ont.ROOTS:
            u[root] = 1.0

        out[protein_id] = dict(sorted(u.items(), key=lambda x: (-x[1], x[0]))[:topk])
    return out


def prediction_stats(preds):
    row_counts = [len(v) for v in preds.values()] or [0]
    scores = [s for terms in preds.values() for s in terms.values()]
    return {
        "proteins": len(preds),
        "rows": int(sum(row_counts)),
        "terms_per_protein_mean": float(np.mean(row_counts)),
        "terms_per_protein_median": float(np.median(row_counts)),
        "score_mean": float(np.mean(scores)) if scores else 0.0,
        "score_p50": float(np.median(scores)) if scores else 0.0,
        "score_p95": float(np.quantile(scores, 0.95)) if scores else 0.0,
    }


def run_combo_grid(goa, pt5, ont, seqs, codec):
    """
    Tests deterministic combinations. Without labels, selection is conservative:
    choose the first robust setting from the defined priority order.
    All combo outputs and stats are written for inspection.
    """
    combo_grid = [
        {"name": "glyph_combo_A_safe", "w_goa": 0.55, "alpha": 0.70, "power": 0.80, "max_s": 0.93, "topk": 270, "glyph_weight": 0.012},
        {"name": "glyph_combo_B_goa_plus", "w_goa": 0.60, "alpha": 0.70, "power": 0.80, "max_s": 0.93, "topk": 270, "glyph_weight": 0.010},
        {"name": "glyph_combo_C_pt5_plus", "w_goa": 0.50, "alpha": 0.68, "power": 0.82, "max_s": 0.93, "topk": 270, "glyph_weight": 0.012},
        {"name": "glyph_combo_D_no_glyph", "w_goa": 0.55, "alpha": 0.70, "power": 0.80, "max_s": 0.93, "topk": 270, "glyph_weight": 0.000},
    ]

    os.makedirs("glyph_combo_outputs", exist_ok=True)
    reports = []
    outputs = {}

    for cfg in combo_grid:
        base = ensemble(goa, pt5, w_goa=cfg["w_goa"])
        cal = glyph_calibrate(base, seqs, codec, glyph_weight=cfg["glyph_weight"])
        final = propagate(
            cal,
            ont,
            alpha=cfg["alpha"],
            power=cfg["power"],
            max_s=cfg["max_s"],
            topk=cfg["topk"],
        )
        outputs[cfg["name"]] = final

        stats = prediction_stats(final)
        report = dict(cfg)
        report.update(stats)
        reports.append(report)

        save_preds(final, f"glyph_combo_outputs/{cfg['name']}.tsv")

    # Conservative default: first combo is intentionally the selected priority.
    selected = combo_grid[0]["name"]
    return selected, outputs[selected], reports


# ----------------------------
# Main run
# ----------------------------

# Prefer original known paths; fall back to discovery.
COMP = "/kaggle/input/cafa-6-protein-function-prediction"
PRED = "/kaggle/input/cafa6-goa-predictions"

obo_path = f"{COMP}/Train/go-basic.obo"
if not Path(obo_path).exists():
    obo_path = find_first("/kaggle/input", names=["go-basic.obo", "go-basic.obo.gz"], suffixes=(".obo", ".obo.gz"))

goa_path = f"{PRED}/goa_submission.tsv"
if not Path(goa_path).exists():
    goa_path = find_first("/kaggle/input", contains=["goa"], suffixes=(".tsv", ".txt", ".tsv.gz"))

pt5_path = f"{PRED}/prott5_interpro_predictions.tsv"
if not Path(pt5_path).exists():
    pt5_path = (
        find_first("/kaggle/input", contains=["prott5", "interpro"], suffixes=(".tsv", ".txt", ".tsv.gz"))
        or find_first("/kaggle/input", contains=["interpro"], suffixes=(".tsv", ".txt", ".tsv.gz"))
        or find_first("/kaggle/input", contains=["prott5"], suffixes=(".tsv", ".txt", ".tsv.gz"))
    )

test_fasta_path = discover_test_fasta()

print("[PATH] OBO:", obo_path)
print("[PATH] GOA:", goa_path)
print("[PATH] PT5:", pt5_path)
print("[PATH] TEST_FASTA:", test_fasta_path)

ont = GOOntology(obo_path)
goa = load_preds(goa_path)
pt5 = load_preds(pt5_path)

codec = ProteinGlyphCodec()
seqs = load_fasta(test_fasta_path) if test_fasta_path else {}

# Keep only sequences relevant to predicted proteins when possible.
predicted_ids = set(goa) | set(pt5)
if seqs and predicted_ids:
    intersect = predicted_ids & set(seqs)
    if intersect:
        seqs = {pid: seqs[pid] for pid in sorted(intersect)}
        print(f"[SEQ] Using predicted/test intersection: {len(seqs):,} proteins")
    else:
        print("[WARN] FASTA loaded, but no IDs intersect prediction files; glyph calibration disabled.")
        seqs = {}

# Roundtrip glyphstring combo tests.
roundtrip_report = {"tested_proteins": 0, "roundtrip_pass": True, "failure_count": 0}
glyph_sample = pd.DataFrame()

if seqs:
    roundtrip_report, glyph_sample = roundtrip_combo_test(seqs, codec, sample_limit=1000, chunk_options=(3, 5, 8))
    print("[GLYPH ROUNDTRIP]", json.dumps(roundtrip_report, indent=2)[:1000])

    if not roundtrip_report["roundtrip_pass"]:
        raise RuntimeError(f"Glyph roundtrip failed: {roundtrip_report['failures_sample']}")

    glyph_sample.head(200).to_csv("glyphstrings_sample.tsv", sep="\t", index=False)

    features = build_glyph_feature_table(seqs, codec)
    features.to_csv("glyph_features.tsv", sep="\t", index=False)
    print("[GLYPH] Wrote glyphstrings_sample.tsv and glyph_features.tsv")
else:
    print("[GLYPH] No matching sequences found. Running base GO combo without glyph calibration.")

selected_name, final, combo_reports = run_combo_grid(goa, pt5, ont, seqs, codec)

# Main competition file.
save_preds(final, "submission.tsv")

audit = {
    "selected_combo": selected_name,
    "roundtrip_report": roundtrip_report,
    "combo_reports": combo_reports,
    "paths": {
        "obo": obo_path,
        "goa": goa_path,
        "pt5": pt5_path,
        "test_fasta": test_fasta_path,
    },
    "final_stats": prediction_stats(final),
    "notes": [
        "Protein sequences are translated to reversible Braille glyphs.",
        "Glyphs are grouped into glyphstrings.",
        "Glyphstrings are translated back to protein sequences before feature extraction.",
        "Final submission.tsv contains no glyphs; only CAFA protein_id, GO term, score rows.",
        "Glyph calibration is low-amplitude and never invents GO terms."
    ],
}
with open("glyph_combo_report.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

print("[DONE] selected_combo:", selected_name)
print("[DONE] final_stats:", json.dumps(audit["final_stats"], indent=2))
print("[DONE] files: submission.tsv, glyph_combo_report.json, glyphstrings_sample.tsv, glyph_features.tsv, glyph_combo_outputs/*.tsv")
